# P2
## Parte 1:

In [3]:
pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.4.7-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (215 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [requests]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pymongo

  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.7 MB/s  0:00:009.5 MB/s eta 0:00:01
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]━━━ 1/2 [pymongo]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
from pymongo import MongoClient

class MongoDBHandler:
    def __init__(self):
        self.mongo_uri = "mongodb://localhost:27017/"
        self.db_name = "pucp_store"
        self.categories_name = "categorias"
        self.productos_name = "productos"
        self.connect_mongo()

    def connect_mongo(self):
        try:
            client = MongoClient(self.mongo_uri)
            self.db = client[self.db_name] if client else None
            print(f"Conectado a la base de datos '{self.db_name}' en MongoDB.")
        except Exception as e:
            print(f"Error al conectar a MongoDB: {e}")

    def fetch_json_data(self, url):
        try:
            response = requests.get(url)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.HTTPError as errh:
            print("HTTP Error:", errh)
        except requests.exceptions.ConnectionError as errc:
            print("Error de conexión:", errc)
        except requests.exceptions.Timeout as errt:
            print("Timeout Error:", errt)
        except requests.exceptions.RequestException as err:
            print("Error desconocido:", err)
        return None

    def fetch_all_categories(self, base_url):
        data = self.fetch_json_data(base_url)
        if not data:
            print("No se pudo obtener las categorías.")
            return 0
        col = self.db[self.categories_name]
        col.delete_many({})
        col.insert_many(data)
        print(f"Se cargaron {len(data)} categorías en '{self.categories_name}'.")
        return len(data)

    def fetch_all_products(self):
        categorias = list(self.db[self.categories_name].find({}, {"_id": 0}))
        if not categorias:
            print("No hay categorías cargadas. Ejecuta fetch_all_categories primero.")
            return 0
        col = self.db[self.productos_name]
        col.delete_many({})
        total = 0
        for cat in categorias:
            url = f"{cat['url']}?limit=0"
            data = self.fetch_json_data(url)
            if not data or not data.get("products"):
                print(f"  - {cat['slug']}: sin productos")
                continue
            productos = data["products"]
            col.insert_many(productos)
            total += len(productos)
            print(f"  - {cat['slug']}: {len(productos)} productos")
        print(f"Total de productos cargados: {total}")
        return total

    # === Parte 2: Operaciones CRUD y filtrado personalizado ===

    def crear_indice(self, campo):
        col = self.db[self.productos_name]
        nombre_indice = col.create_index(campo)
        print(f"Indice creado en el campo '{campo}': {nombre_indice}")
        return nombre_indice

    def crear_producto(self, producto):
        col = self.db[self.productos_name]
        resultado = col.insert_one(producto)
        print(f"Producto creado con _id: {resultado.inserted_id}")
        return resultado.inserted_id

    def obtener_productos(self, filtro=None):
        col = self.db[self.productos_name]
        return list(col.find(filtro or {}))

    def obtener_producto(self, filtro):
        col = self.db[self.productos_name]
        return col.find_one(filtro)

    def actualizar_producto(self, filtro, cambios):
        col = self.db[self.productos_name]
        resultado = col.update_one(filtro, {"$set": cambios})
        print(f"Documentos encontrados: {resultado.matched_count}, modificados: {resultado.modified_count}")
        return resultado.modified_count

    def eliminar_producto(self, filtro):
        col = self.db[self.productos_name]
        resultado = col.delete_one(filtro)
        print(f"Documentos eliminados: {resultado.deleted_count}")
        return resultado.deleted_count

    def obtener_productos_por_precio(self, precio):
        col = self.db[self.productos_name]
        query = {"$or": [{"precio": precio}, {"price": precio}]}
        return list(col.find(query))

    def obtener_productos_por_nombre(self, texto):
        col = self.db[self.productos_name]
        regex = {"$regex": texto, "$options": "i"}
        query = {"$or": [{"nombre": regex}, {"title": regex}]}
        return list(col.find(query))

    def precio_promedio(self):
        col = self.db[self.productos_name]
        pipeline = [
            {"$project": {"precio_unico": {"$ifNull": ["$precio", "$price"]}}},
            {"$group": {"_id": None, "promedio": {"$avg": "$precio_unico"}}}
        ]
        resultado = list(col.aggregate(pipeline))
        return resultado[0]["promedio"] if resultado else None

    def contar_productos(self):
        col = self.db[self.productos_name]
        return col.count_documents({})

    def mayor_stock_categoria(self):
        col = self.db[self.productos_name]
        pipeline = [
            {"$match": {"stock": {"$exists": True}, "category": {"$exists": True}}},
            {"$sort": {"stock": -1}},
            {"$group": {
                "_id": "$category",
                "producto": {"$first": "$title"},
                "stock": {"$first": "$stock"}
            }},
            {"$sort": {"_id": 1}}
        ]
        return list(col.aggregate(pipeline))


In [4]:
mongo_handler = MongoDBHandler()

categories_url = "https://dummyjson.com/products/categories"

mongo_handler.fetch_all_categories(categories_url)
mongo_handler.fetch_all_products()

Conectado a la base de datos 'pucp_store' en MongoDB.
Se cargaron 24 categorías en 'categorias'.
  - beauty: 5 productos
  - fragrances: 5 productos
  - furniture: 5 productos
  - groceries: 27 productos
  - home-decoration: 5 productos
  - kitchen-accessories: 30 productos
  - laptops: 5 productos
  - mens-shirts: 5 productos
  - mens-shoes: 5 productos
  - mens-watches: 6 productos
  - mobile-accessories: 14 productos
  - motorcycle: 5 productos
  - skin-care: 3 productos
  - smartphones: 16 productos
  - sports-accessories: 17 productos
  - sunglasses: 5 productos
  - tablets: 3 productos
  - tops: 5 productos
  - vehicle: 5 productos
  - womens-bags: 5 productos
  - womens-dresses: 5 productos
  - womens-jewellery: 3 productos
  - womens-shoes: 5 productos
  - womens-watches: 5 productos
Total de productos cargados: 194


194

In [5]:
print("Categorías en DB:", mongo_handler.db[mongo_handler.categories_name].count_documents({}))
print("Productos en DB:", mongo_handler.db[mongo_handler.productos_name].count_documents({}))
print("\nEjemplo de categoría:")
print(mongo_handler.db[mongo_handler.categories_name].find_one({}, {"_id": 0}))
print("\nEjemplo de producto (solo algunos campos):")
print(mongo_handler.db[mongo_handler.productos_name].find_one({}, {"_id": 0, "title": 1, "category": 1, "price": 1, "stock": 1}))

Categorías en DB: 24
Productos en DB: 194

Ejemplo de categoría:
{'slug': 'beauty', 'name': 'Beauty', 'url': 'https://dummyjson.com/products/category/beauty'}

Ejemplo de producto (solo algunos campos):
{'title': 'Essence Mascara Lash Princess', 'category': 'beauty', 'price': 9.99, 'stock': 99}


## Parte 2: Operaciones CRUD y filtrado personalizado

Se agregan a `MongoDBHandler` los metodos de creacion de indice, CRUD basico sobre `productos`,
consultas de filtrado por precio/nombre y consultas agregadas (promedio de precio, conteo,
producto con mayor stock por categoria).

In [6]:
mongo_handler = MongoDBHandler()

# Crear indice en campo especifico
mongo_handler.crear_indice(campo="nombre")

# Crear productos
mongo_handler.crear_producto({"nombre": "Producto 1", "precio": 100})
mongo_handler.crear_producto({"nombre": "Producto 2", "precio": 200})
mongo_handler.crear_producto({"nombre": "Producto 3", "precio": 300})


Conectado a la base de datos 'pucp_store' en MongoDB.
Indice creado en el campo 'nombre': nombre_1
Producto creado con _id: 6a471341e267b3517281a811
Producto creado con _id: 6a471341e267b3517281a812
Producto creado con _id: 6a471341e267b3517281a813


ObjectId('6a471341e267b3517281a813')

In [7]:
# Leer productos
productos = mongo_handler.obtener_productos()
print("Cantidad de productos en la coleccion:", len(productos))
print("Primeros 3 productos:", productos[:3])


Cantidad de productos en la coleccion: 197
Primeros 3 productos: [{'_id': ObjectId('6a471336e267b3517281a74e'), 'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.053Z

In [8]:
# Leer producto especifico
producto = mongo_handler.obtener_producto({"_id": productos[0]["_id"]})
print("Producto obtenido:", producto)


Producto obtenido: {'_id': ObjectId('6a471336e267b3517281a74e'), 'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Lucas Gordon', 'reviewerEma

In [9]:
# Actualizar un producto
if productos:
    mongo_handler.actualizar_producto({"_id": productos[0]["_id"]}, {"precio": 120})
    print("Producto tras actualizar:", mongo_handler.obtener_producto({"_id": productos[0]["_id"]}))


Documentos encontrados: 1, modificados: 1
Producto tras actualizar: {'_id': ObjectId('6a471336e267b3517281a74e'), 'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.05

In [10]:
# Eliminar un producto
if productos:
    mongo_handler.eliminar_producto({"_id": productos[0]["_id"]})
    print("Total tras eliminar:", mongo_handler.contar_productos())


Documentos eliminados: 1
Total tras eliminar: 196


In [11]:
# Consultas basicas
print("Productos con precio 100:", mongo_handler.obtener_productos_por_precio(100))
print("Productos que contienen 'iPhone' en el nombre/titulo:",
      mongo_handler.obtener_productos_por_nombre("iPhone"))


Productos con precio 100: [{'_id': ObjectId('6a471341e267b3517281a811'), 'nombre': 'Producto 1', 'precio': 100}]
Productos que contienen 'iPhone' en el nombre/titulo: [{'_id': ObjectId('6a47133be267b3517281a7b5'), 'id': 104, 'title': 'Apple iPhone Charger', 'description': 'The Apple iPhone Charger is a high-quality charger designed for fast and efficient charging of your iPhone. Ensure your device stays powered up and ready to go.', 'category': 'mobile-accessories', 'price': 19.99, 'discountPercentage': 18.52, 'rating': 4.15, 'stock': 31, 'tags': ['electronics', 'chargers'], 'brand': 'Apple', 'sku': 'MOB-APP-APP-104', 'weight': 1, 'dimensions': {'width': 13.63, 'height': 26.25, 'depth': 5.95}, 'warrantyInformation': '1 year warranty', 'shippingInformation': 'Ships in 2 weeks', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 1, 'comment': 'Very disappointed!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Sadie Morales', 'reviewerEmail': 'sadie.morales@x.dummyjson.com'},

In [12]:
# Consultas agregadas
print("Precio promedio de productos:", mongo_handler.precio_promedio())
print("Total de productos:", mongo_handler.contar_productos())
print("El producto con mayor stock por categoria:")
for doc in mongo_handler.mayor_stock_categoria():
    print(" -", doc)


Precio promedio de productos: 1557.0916836734693
Total de productos: 196
El producto con mayor stock por categoria:
 - {'_id': 'beauty', 'producto': 'Red Lipstick', 'stock': 91}
 - {'_id': 'fragrances', 'producto': "Dior J'adore", 'stock': 98}
 - {'_id': 'furniture', 'producto': 'Annibale Colombo Bed', 'stock': 88}
 - {'_id': 'groceries', 'producto': 'Kiwi', 'stock': 99}
 - {'_id': 'home-decoration', 'producto': 'Family Tree Photo Frame', 'stock': 77}
 - {'_id': 'kitchen-accessories', 'producto': 'Lunch Box', 'stock': 94}
 - {'_id': 'laptops', 'producto': 'Huawei Matebook X Pro', 'stock': 75}
 - {'_id': 'mens-shirts', 'producto': 'Men Check Shirt', 'stock': 95}
 - {'_id': 'mens-shoes', 'producto': 'Puma Future Rider Trainers', 'stock': 90}
 - {'_id': 'mens-watches', 'producto': 'Longines Master Collection', 'stock': 100}
 - {'_id': 'mobile-accessories', 'producto': 'iPhone 12 Silicone Case with MagSafe Plum', 'stock': 69}
 - {'_id': 'motorcycle', 'producto': 'Scooter Motorcycle', 'stoc